# Amaliy mashgʻulot 10. Sintaktik tahlil: bogʻliqlik daraxtlari va tarkibiy qismlar daraxtlari

10-laboratoriyada bogʻliqlik yoylarini koʻrdingiz, arc-standard parserini qadamma-qadam ishga tushirdingiz va CYK jadvalini toʻldirdingiz. Bu yerda xuddi shuni kodda bajaramiz:

1. bogʻliqlik daraxtlarini CoNLL-U formatida yozamiz, yoylarni chizamiz va oʻzbek (SOV) hamda rus (SVO) tillaridagi soʻz tartibini solishtiramiz;
2. arc-standard oʻtishlar tizimini amalga oshiramiz: orakul, stek va bufer izi, keyin amallar klassifikatorini oʻqitib, haqiqiy (kichik boʻlsa ham) parser hosil qilamiz;
3. Xomskiy normal shaklidagi grammatika uchun CYK algoritmini yozamiz va koʻp maʼnoli «eski kitob javoni» birikmasining barcha daraxtlarini tiklaymiz;
4. natijani `nltk.ChartParser` bilan solishtiramiz, daraxtlarni `nltk.Tree` orqali chiqaramiz va ehtimolli grammatika (PCFG) bilan eng yaxshi daraxtni tanlaymiz.

*«NLP algoritmlari» kursining 10-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Bogʻliqlik daraxtlari va CoNLL-U formati

Bogʻliqlik daraxtida ildizdan boshqa har bir soʻz aynan bitta soʻzga — oʻz bosh soʻziga (head) tobe boʻladi. Yoyda bogʻlanish turi yoziladi (Universal Dependencies): `nsubj` — ega, `obj` — vositasiz toʻldiruvchi, `obl` — kelishik yoki koʻmakchili hol/toʻldiruvchi, `amod` — sifatlovchi, `nmod:poss` — qaratqich kelishigidagi aniqlovchi, `case` — koʻmakchi, `advmod` — ravish.

CoNLL-U formati: har qatorda bitta soʻz, 10 ta ustun: ID, FORM (soʻz shakli), LEMMA, UPOS, XPOS, FEATS, HEAD (bosh soʻz raqami, 0 — ildiz), DEPREL, DEPS, MISC. Quyida shu amaliy mashgʻulot uchun belgilangan kichik treebank: 19 ta oʻqitish va 6 ta test gapi. Ishlatilmaydigan ustunlar `_` bilan toʻldirilgan. Soʻz tartibini solishtirish uchun xuddi shu gaplar rus tilida ham berilgan.

UD qoidalariga koʻra oʻzbek tilida kelishik qoʻshimchasi soʻzning bir qismi: «kutubxonada» bitta token, uning kelishigi `obl` turini koʻrsatadi. Koʻmakchi esa («bilan») alohida token.

Quyida treebank oʻqiladi va birinchi gap (laboratoriyadagi misol) uning ruscha tarjimasi «Студент читает интересную книгу в библиотеке» bilan birga chiziladi.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

TRAIN_CONLLU = """
# sent_id = 1
# text = Talaba kutubxonada qiziqarli kitob oʻqiyapti
1  Talaba       talaba     NOUN  _ _ 5  nsubj     _ _
2  kutubxonada  kutubxona  NOUN  _ _ 5  obl       _ _
3  qiziqarli    qiziqarli  ADJ   _ _ 4  amod      _ _
4  kitob        kitob      NOUN  _ _ 5  obj       _ _
5  oʻqiyapti    oʻqi       VERB  _ _ 0  root      _ _

# sent_id = 2
# text = Onam mazali palov pishiryapti
1  Onam         ona     NOUN  _ _ 4  nsubj     _ _
2  mazali       mazali  ADJ   _ _ 3  amod      _ _
3  palov        palov   NOUN  _ _ 4  obj       _ _
4  pishiryapti  pishir  VERB  _ _ 0  root      _ _

# sent_id = 3
# text = Akam Toshkentda yashaydi
1  Akam        aka       NOUN  _ _ 3  nsubj     _ _
2  Toshkentda  Toshkent  PROPN _ _ 3  obl       _ _
3  yashaydi    yasha     VERB  _ _ 0  root      _ _

# sent_id = 4
# text = Bolalar hovlida oʻynayapti
1  Bolalar     bola   NOUN  _ _ 3  nsubj     _ _
2  hovlida     hovli  NOUN  _ _ 3  obl       _ _
3  oʻynayapti  oʻyna  VERB  _ _ 0  root      _ _

# sent_id = 5
# text = Oʻqituvchi oʻquvchilarga yangi kitoblar berdi
1  Oʻqituvchi     oʻqituvchi  NOUN  _ _ 5  nsubj     _ _
2  oʻquvchilarga  oʻquvchi    NOUN  _ _ 5  obl       _ _
3  yangi          yangi       ADJ   _ _ 4  amod      _ _
4  kitoblar       kitob       NOUN  _ _ 5  obj       _ _
5  berdi          ber         VERB  _ _ 0  root      _ _

# sent_id = 6
# text = Men kecha yangi mashina oldim
1  Men      men      PRON  _ _ 5  nsubj     _ _
2  kecha    kecha    ADV   _ _ 5  advmod    _ _
3  yangi    yangi    ADJ   _ _ 4  amod      _ _
4  mashina  mashina  NOUN  _ _ 5  obj       _ _
5  oldim    ol       VERB  _ _ 0  root      _ _

# sent_id = 7
# text = Singlim dugonasiga xat yozyapti
1  Singlim     singil  NOUN  _ _ 4  nsubj     _ _
2  dugonasiga  dugona  NOUN  _ _ 4  obl       _ _
3  xat         xat     NOUN  _ _ 4  obj       _ _
4  yozyapti    yoz     VERB  _ _ 0  root      _ _

# sent_id = 8
# text = Otam katta zavodda ishlaydi
1  Otam      ota    NOUN  _ _ 4  nsubj     _ _
2  katta     katta  ADJ   _ _ 3  amod      _ _
3  zavodda   zavod  NOUN  _ _ 4  obl       _ _
4  ishlaydi  ishla  VERB  _ _ 0  root      _ _

# sent_id = 9
# text = Qushlar janubga uchyapti
1  Qushlar   qush   NOUN  _ _ 3  nsubj     _ _
2  janubga   janub  NOUN  _ _ 3  obl       _ _
3  uchyapti  uch    VERB  _ _ 0  root      _ _

# sent_id = 10
# text = Men doʻstim bilan parkka bordim
1  Men      men    PRON  _ _ 5  nsubj     _ _
2  doʻstim  doʻst  NOUN  _ _ 5  obl       _ _
3  bilan    bilan  ADP   _ _ 2  case      _ _
4  parkka   park   NOUN  _ _ 5  obl       _ _
5  bordim   bor    VERB  _ _ 0  root      _ _

# sent_id = 11
# text = Buvim bolalarga qiziqarli ertak aytdi
1  Buvim      buvi       NOUN  _ _ 5  nsubj     _ _
2  bolalarga  bola       NOUN  _ _ 5  obl       _ _
3  qiziqarli  qiziqarli  ADJ   _ _ 4  amod      _ _
4  ertak      ertak      NOUN  _ _ 5  obj       _ _
5  aytdi      ayt        VERB  _ _ 0  root      _ _

# sent_id = 12
# text = Talabalar qiyin imtihon topshirishdi
1  Talabalar     talaba   NOUN  _ _ 4  nsubj     _ _
2  qiyin         qiyin    ADJ   _ _ 3  amod      _ _
3  imtihon       imtihon  NOUN  _ _ 4  obj       _ _
4  topshirishdi  topshir  VERB  _ _ 0  root      _ _

# sent_id = 13
# text = Mushuk divanda uxlayapti
1  Mushuk     mushuk  NOUN  _ _ 3  nsubj     _ _
2  divanda    divan   NOUN  _ _ 3  obl       _ _
3  uxlayapti  uxla    VERB  _ _ 0  root      _ _

# sent_id = 14
# text = Bola masalani tez yechdi
1  Bola      bola    NOUN  _ _ 4  nsubj     _ _
2  masalani  masala  NOUN  _ _ 4  obj       _ _
3  tez       tez     ADV   _ _ 4  advmod    _ _
4  yechdi    yech    VERB  _ _ 0  root      _ _

# sent_id = 15
# text = Biz kitob oʻqiymiz
1  Biz       biz    PRON  _ _ 3  nsubj     _ _
2  kitob     kitob  NOUN  _ _ 3  obj       _ _
3  oʻqiymiz  oʻqi   VERB  _ _ 0  root      _ _

# sent_id = 16
# text = Shifokor kasal bolani koʻrdi
1  Shifokor  shifokor  NOUN  _ _ 4  nsubj     _ _
2  kasal     kasal     ADJ   _ _ 3  amod      _ _
3  bolani    bola      NOUN  _ _ 4  obj       _ _
4  koʻrdi    koʻr      VERB  _ _ 0  root      _ _

# sent_id = 17
# text = Ahmad doʻkondan non oldi
1  Ahmad      Ahmad   PROPN _ _ 4  nsubj     _ _
2  doʻkondan  doʻkon  NOUN  _ _ 4  obl       _ _
3  non        non     NOUN  _ _ 4  obj       _ _
4  oldi       ol      VERB  _ _ 0  root      _ _

# sent_id = 18
# text = Qiz onasiga chiroyli gullar berdi
1  Qiz       qiz       NOUN  _ _ 5  nsubj     _ _
2  onasiga   ona       NOUN  _ _ 5  obl       _ _
3  chiroyli  chiroyli  ADJ   _ _ 4  amod      _ _
4  gullar    gul       NOUN  _ _ 5  obj       _ _
5  berdi     ber       VERB  _ _ 0  root      _ _

# sent_id = 19
# text = Akamning kitobi stolda yotibdi
1  Akamning  aka    NOUN  _ _ 2  nmod:poss _ _
2  kitobi    kitob  NOUN  _ _ 4  nsubj     _ _
3  stolda    stol   NOUN  _ _ 4  obl       _ _
4  yotibdi   yot    VERB  _ _ 0  root      _ _
"""

TEST_CONLLU = """
# sent_id = 1
# text = Akam qiziqarli kitob oldi
1  Akam       aka        NOUN  _ _ 4  nsubj     _ _
2  qiziqarli  qiziqarli  ADJ   _ _ 3  amod      _ _
3  kitob      kitob      NOUN  _ _ 4  obj       _ _
4  oldi       ol         VERB  _ _ 0  root      _ _

# sent_id = 2
# text = Talabalar kutubxonada oʻqiyapti
1  Talabalar    talaba     NOUN  _ _ 3  nsubj     _ _
2  kutubxonada  kutubxona  NOUN  _ _ 3  obl       _ _
3  oʻqiyapti    oʻqi       VERB  _ _ 0  root      _ _

# sent_id = 3
# text = Oʻqituvchi direktorga xat yozdi
1  Oʻqituvchi  oʻqituvchi  NOUN  _ _ 4  nsubj     _ _
2  direktorga  direktor    NOUN  _ _ 4  obl       _ _
3  xat         xat         NOUN  _ _ 4  obj       _ _
4  yozdi       yoz         VERB  _ _ 0  root      _ _

# sent_id = 4
# text = Bolalar sovuq suv ichyapti
1  Bolalar   bola   NOUN  _ _ 4  nsubj     _ _
2  sovuq     sovuq  ADJ   _ _ 3  amod      _ _
3  suv       suv    NOUN  _ _ 4  obj       _ _
4  ichyapti  ich    VERB  _ _ 0  root      _ _

# sent_id = 5
# text = Otam gazetani tez oʻqidi
1  Otam      ota     NOUN  _ _ 4  nsubj     _ _
2  gazetani  gazeta  NOUN  _ _ 4  obj       _ _
3  tez       tez     ADV   _ _ 4  advmod    _ _
4  oʻqidi    oʻqi    VERB  _ _ 0  root      _ _

# sent_id = 6
# text = Doʻstimning singlisi kasalxonada ishlaydi
1  Doʻstimning  doʻst      NOUN  _ _ 2  nmod:poss _ _
2  singlisi     singil     NOUN  _ _ 4  nsubj     _ _
3  kasalxonada  kasalxona  NOUN  _ _ 4  obl       _ _
4  ishlaydi     ishla      VERB  _ _ 0  root      _ _
"""

# xuddi shu gaplar rus tilida — soʻz tartibini solishtirish uchun (1-boʻlim)
OTHER_CONLLU = """
# sent_id = 1
# text = Студент читает интересную книгу в библиотеке
1  Студент     студент     NOUN  _ _ 2  nsubj     _ _
2  читает      читать      VERB  _ _ 0  root      _ _
3  интересную  интересный  ADJ   _ _ 4  amod      _ _
4  книгу       книга       NOUN  _ _ 2  obj       _ _
5  в           в           ADP   _ _ 6  case      _ _
6  библиотеке  библиотека  NOUN  _ _ 2  obl       _ _

# sent_id = 2
# text = Мама готовит вкусный плов
1  Мама     мама      NOUN  _ _ 2  nsubj     _ _
2  готовит  готовить  VERB  _ _ 0  root      _ _
3  вкусный  вкусный   ADJ   _ _ 4  amod      _ _
4  плов     плов      NOUN  _ _ 2  obj       _ _

# sent_id = 3
# text = Мой брат живёт в Ташкенте
1  Мой       мой      DET   _ _ 2  det       _ _
2  брат      брат     NOUN  _ _ 3  nsubj     _ _
3  живёт     жить     VERB  _ _ 0  root      _ _
4  в         в        ADP   _ _ 5  case      _ _
5  Ташкенте  Ташкент  PROPN _ _ 3  obl       _ _

# sent_id = 4
# text = Дети играют во дворе
1  Дети    ребёнок  NOUN  _ _ 2  nsubj     _ _
2  играют  играть   VERB  _ _ 0  root      _ _
3  во      в        ADP   _ _ 4  case      _ _
4  дворе   двор     NOUN  _ _ 2  obl       _ _

# sent_id = 5
# text = Учитель дал ученикам новые книги
1  Учитель   учитель  NOUN  _ _ 2  nsubj     _ _
2  дал       дать     VERB  _ _ 0  root      _ _
3  ученикам  ученик   NOUN  _ _ 2  iobj      _ _
4  новые     новый    ADJ   _ _ 5  amod      _ _
5  книги     книга    NOUN  _ _ 2  obj       _ _

# sent_id = 6
# text = Я вчера купил новую машину
1  Я       я       PRON  _ _ 3  nsubj     _ _
2  вчера   вчера   ADV   _ _ 3  advmod    _ _
3  купил   купить  VERB  _ _ 0  root      _ _
4  новую   новый   ADJ   _ _ 5  amod      _ _
5  машину  машина  NOUN  _ _ 3  obj       _ _

# sent_id = 7
# text = Сестра пишет письмо подруге
1  Сестра   сестра   NOUN  _ _ 2  nsubj     _ _
2  пишет    писать   VERB  _ _ 0  root      _ _
3  письмо   письмо   NOUN  _ _ 2  obj       _ _
4  подруге  подруга  NOUN  _ _ 2  iobj      _ _

# sent_id = 8
# text = Отец работает на большом заводе
1  Отец      отец      NOUN  _ _ 2  nsubj     _ _
2  работает  работать  VERB  _ _ 0  root      _ _
3  на        на        ADP   _ _ 5  case      _ _
4  большом   большой   ADJ   _ _ 5  amod      _ _
5  заводе    завод     NOUN  _ _ 2  obl       _ _

# sent_id = 9
# text = Птицы летят на юг
1  Птицы  птица   NOUN  _ _ 2  nsubj     _ _
2  летят  лететь  VERB  _ _ 0  root      _ _
3  на     на      ADP   _ _ 4  case      _ _
4  юг     юг      NOUN  _ _ 2  obl       _ _

# sent_id = 10
# text = Я пошёл в парк с другом
1  Я       я      PRON  _ _ 2  nsubj     _ _
2  пошёл   пойти  VERB  _ _ 0  root      _ _
3  в       в      ADP   _ _ 4  case      _ _
4  парк    парк   NOUN  _ _ 2  obl       _ _
5  с       с      ADP   _ _ 6  case      _ _
6  другом  друг   NOUN  _ _ 2  obl       _ _

# sent_id = 11
# text = Бабушка рассказала детям интересную сказку
1  Бабушка     бабушка     NOUN  _ _ 2  nsubj     _ _
2  рассказала  рассказать  VERB  _ _ 0  root      _ _
3  детям       ребёнок     NOUN  _ _ 2  iobj      _ _
4  интересную  интересный  ADJ   _ _ 5  amod      _ _
5  сказку      сказка      NOUN  _ _ 2  obj       _ _

# sent_id = 12
# text = Студенты сдали трудный экзамен
1  Студенты  студент  NOUN  _ _ 2  nsubj     _ _
2  сдали     сдать    VERB  _ _ 0  root      _ _
3  трудный   трудный  ADJ   _ _ 4  amod      _ _
4  экзамен   экзамен  NOUN  _ _ 2  obj       _ _

# sent_id = 13
# text = Кошка спит на диване
1  Кошка   кошка  NOUN  _ _ 2  nsubj     _ _
2  спит    спать  VERB  _ _ 0  root      _ _
3  на      на     ADP   _ _ 4  case      _ _
4  диване  диван  NOUN  _ _ 2  obl       _ _

# sent_id = 14
# text = Мальчик быстро решил задачу
1  Мальчик  мальчик  NOUN  _ _ 3  nsubj     _ _
2  быстро   быстро   ADV   _ _ 3  advmod    _ _
3  решил    решить   VERB  _ _ 0  root      _ _
4  задачу   задача   NOUN  _ _ 3  obj       _ _

# sent_id = 15
# text = Мы читаем книгу
1  Мы      мы      PRON  _ _ 2  nsubj     _ _
2  читаем  читать  VERB  _ _ 0  root      _ _
3  книгу   книга   NOUN  _ _ 2  obj       _ _

# sent_id = 16
# text = Врач осмотрел больного ребёнка
1  Врач      врач       NOUN  _ _ 2  nsubj     _ _
2  осмотрел  осмотреть  VERB  _ _ 0  root      _ _
3  больного  больной    ADJ   _ _ 4  amod      _ _
4  ребёнка   ребёнок    NOUN  _ _ 2  obj       _ _

# sent_id = 17
# text = Ахмад купил хлеб в магазине
1  Ахмад     Ахмад    PROPN _ _ 2  nsubj     _ _
2  купил     купить   VERB  _ _ 0  root      _ _
3  хлеб      хлеб     NOUN  _ _ 2  obj       _ _
4  в         в        ADP   _ _ 5  case      _ _
5  магазине  магазин  NOUN  _ _ 2  obl       _ _

# sent_id = 18
# text = Девочка подарила маме красивые цветы
1  Девочка   девочка   NOUN  _ _ 2  nsubj     _ _
2  подарила  подарить  VERB  _ _ 0  root      _ _
3  маме      мама      NOUN  _ _ 2  iobj      _ _
4  красивые  красивый  ADJ   _ _ 5  amod      _ _
5  цветы     цветок    NOUN  _ _ 2  obj       _ _

# sent_id = 19
# text = Книга моего брата лежит на столе
1  Книга  книга   NOUN  _ _ 4  nsubj     _ _
2  моего  мой     DET   _ _ 3  det       _ _
3  брата  брат    NOUN  _ _ 1  nmod      _ _
4  лежит  лежать  VERB  _ _ 0  root      _ _
5  на     на      ADP   _ _ 6  case      _ _
6  столе  стол    NOUN  _ _ 4  obl       _ _
"""

def read_conllu(text):
    # bloklar boʻsh qator bilan ajratilgan; # bilan boshlangan qatorlar — izohlar; ustunlar boʻshliq bilan ajratilgan
    sents = []
    for block in text.strip().split("\n\n"):
        toks = []
        for line in block.splitlines():
            if line.startswith("#") or not line.strip():
                continue
            c = line.split()
            toks.append(dict(id=int(c[0]), form=c[1], lemma=c[2], upos=c[3], head=int(c[6]), deprel=c[7]))
        sents.append(toks)
    return sents

train, test, other = read_conllu(TRAIN_CONLLU), read_conllu(TEST_CONLLU), read_conllu(OTHER_CONLLU)
print(f"oʻqitish: {len(train)} gap, test: {len(test)} gap\n")
print(pd.DataFrame(train[0])[["id", "form", "lemma", "upos", "head", "deprel"]].to_string(index=False))


def draw_tree(sent, ax, title=""):
    # yoy bosh soʻzdan tobe soʻzga boradi; qizil — bosh soʻz oʻngda, koʻk — bosh soʻz chapda
    n = len(sent)
    top = 0.6
    for t in sent:
        ax.text(t["id"], 0, t["form"], ha="center", va="top", fontsize=11)
        ax.text(t["id"], -0.45, t["upos"], ha="center", va="top", fontsize=8, color="gray")
    for t in sent:
        d, h = t["id"], t["head"]
        if h == 0:
            continue
        height = 0.3 + 0.45 * abs(h - d)
        top = max(top, height)
        theta = np.linspace(0, np.pi, 60)
        xs = (h + d) / 2 + (h - d) / 2 * np.cos(theta)
        ys = 0.15 + height * np.sin(theta)
        col = "#C0392B" if h > d else "#2B55C9"
        ax.plot(xs, ys, color=col, lw=1.5)
        ax.annotate("", xy=(d, 0.15), xytext=(xs[-4], ys[-4]), arrowprops=dict(arrowstyle="->", color=col, lw=1.5))
        ax.text((h + d) / 2, 0.15 + height, t["deprel"], ha="center", va="center", fontsize=9, color=col,
                bbox=dict(fc="white", ec="none", pad=0.5))
    root = [t["id"] for t in sent if t["head"] == 0][0]
    ax.annotate("", xy=(root, 0.15), xytext=(root, top + 0.6), arrowprops=dict(arrowstyle="->", color="black"))
    ax.text(root, top + 0.65, "root", ha="center", fontsize=9)
    ax.set_xlim(0.4, n + 0.6)
    ax.set_ylim(-0.9, top + 1.0)
    ax.axis("off")
    ax.set_title(title, fontsize=10)

fig, axes = plt.subplots(2, 1, figsize=(9, 6.5))
draw_tree(train[0], axes[0], "oʻzbek")
draw_tree(other[0], axes[1], "rus")
axes[0].plot([], [], color="#C0392B", label="bosh soʻz oʻngda")
axes[0].plot([], [], color="#2B55C9", label="bosh soʻz chapda")
fig.legend(loc="lower center", ncol=2, fontsize=9)
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

Oʻzbek gapida feʼl oxirida turadi (SOV tartibi) va barcha yoylar oʻngdan chapga yoʻnalgan: til izchil ravishda **bosh soʻzi oxirida** (head-final). Rus gapida feʼl oʻrtada: ega chapda, toʻldiruvchi va hol oʻngda. Buni butun treebank boʻyicha hisoblaymiz: har bir bogʻlanish turi uchun bosh soʻzdan oldin turgan tobe soʻzlar ulushi.

Shu bilan birga proektivlikni tekshiramiz: agar yoylarni gap ustida kesishmasdan chizish mumkin boʻlsa, daraxt proektiv. 2-boʻlimdagi arc-standard usuli faqat proektiv daraxtlarni quradi.

In [ ]:
def ancestors(sent, i):
    head = {t["id"]: t["head"] for t in sent}
    out = set()
    while i != 0:
        i = head[i]
        out.add(i)
    return out

def is_projective(sent):
    # (h, d) yoyi proektiv, agar h va d orasidagi barcha soʻzlar h ga (bevosita yoki bilvosita) tobe boʻlsa
    for t in sent:
        h, d = t["head"], t["id"]
        for k in range(min(h, d) + 1, max(h, d)):
            if h not in ancestors(sent, k):
                return False
    return True

bad = [" ".join(t["form"] for t in s) for s in train + test + other if not is_projective(s)]
print("barcha daraxtlar proektiv" if not bad else "proektiv emas: " + "; ".join(bad))

def direction_table(sents):
    rows = [(t["deprel"], t["id"] < t["head"]) for s in sents for t in s if t["head"] != 0]
    df = pd.DataFrame(rows, columns=["deprel", "before"])
    tab = df.groupby("deprel")["before"].agg(["mean", "size"])
    tab.loc["hammasi"] = [df["before"].mean(), len(df)]
    return tab

tabs = {"oʻzbek": direction_table(train + test), "rus": direction_table(other)}
order = sorted((set(tabs["oʻzbek"].index) | set(tabs["rus"].index)) - {"hammasi"}) + ["hammasi"]
summary = pd.DataFrame({lang: [f"{tab.loc[r, 'mean']:.2f}  (n={int(tab.loc[r, 'size'])})" if r in tab.index else "—"
                               for r in order] for lang, tab in tabs.items()}, index=order)
print("\noʻz bosh soʻzidan OLDIN turgan tobe soʻzlar ulushi:")
print(summary.to_string())

rels = ["nsubj", "obj", "obl", "amod", "case", "hammasi"]
x = np.arange(len(rels))
plt.figure(figsize=(7, 3.2))
for k, (lang, tab) in enumerate(tabs.items()):
    vals = [tab["mean"].get(r, np.nan) for r in rels]
    plt.bar(x + (k - 0.5) * 0.38, vals, 0.38, label=lang, color=["#2B55C9", "#E08E2B"][k])
plt.xticks(x, rels)
plt.ylim(0, 1.05)
plt.ylabel("ulush")
plt.title("Tobe soʻz qayerda turadi: «bosh soʻzdan oldin» holatlar ulushi", fontsize=10)
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

Raqamlar tipologiyani tasdiqlaydi. Oʻzbek tilida deyarli hamma narsa bosh soʻzdan oldin turadi; istisno — `case`: `bilan` koʻmakchisi otdan keyin keladi, UD da esa yordamchi soʻz tobe hisoblanadi. Rus tilida `obj` va `obl` feʼldan keyin, predloglar (`case`) esa otdan oldin turadi. Sifatlar (`amod`) ikkala tilda ham otdan oldin keladi.

`nltk` CoNLL-U ni `DependencyGraph` obyektiga oʻqiy oladi va daraxtni chop etadi:

In [ ]:
from nltk.parse import DependencyGraph

def to_conllu_text(sent, heads=None):
    # tabulyatsiya bilan ajratilgan 10 ta CoNLL-U ustuni; heads — {id: (bosh soʻz, tur)} lugʻati, agar daraxtni parser bashorat qilgan boʻlsa
    lines = []
    for t in sent:
        h, r = heads[t["id"]] if heads else (t["head"], t["deprel"])
        lines.append("\t".join([str(t["id"]), t["form"], t["lemma"], t["upos"], "_", "_", str(h), r, "_", "_"]))
    return "\n".join(lines)

g = DependencyGraph(to_conllu_text(train[0]), top_relation_label="root")
g.tree().pretty_print()
for (h, h_pos), rel, (d, d_pos) in g.triples():
    print(f"   {h:<14} --{rel}--> {d}")

## 2. Arc-standard parseri

Oʻtishlarga asoslangan (transition-based) parser gapni chapdan oʻngga oʻqiydi va ishlangan soʻzlar **stekini** hamda hali oʻqilmagan soʻzlar **buferini** saqlaydi. Har qadamda quyidagi amallardan birini bajaradi:

- **SHIFT** — buferning birinchi soʻzini stekka oʻtkazish;
- **LEFT-ARC:r** — stekdagi yuqoridan ikkinchi soʻz yuqoridagisiga tobe boʻladi (chapga yoy), uni stekdan olib tashlaymiz;
- **RIGHT-ARC:r** — yuqoridagi soʻz ikkinchisiga tobe boʻladi (oʻngga yoy), yuqoridagisini olib tashlaymiz.

**Orakul** etalon daraxtga qarab qaysi amal toʻgʻriligini aytadi: ikkinchi soʻz yuqoridagiga tobe boʻlsa — LEFT-ARC; yuqoridagi ikkinchisiga tobe boʻlsa va oʻzining barcha tobe soʻzlarini yigʻib boʻlgan boʻlsa — RIGHT-ARC; aks holda SHIFT. $n$ soʻzli gap uchun amallar aynan $2n$ ta: har bir soʻz bir marta stekka tushadi va bir marta undan olinadi.

In [ ]:
def apply(action, stack, buffer, arcs):
    stack, buffer, arcs = list(stack), list(buffer), list(arcs)
    if action == "SHIFT":
        stack.append(buffer.pop(0))
    elif action.startswith("LEFT-ARC"):
        dep = stack.pop(-2)
        arcs.append((stack[-1], dep, action.split(":", 1)[1]))
    else:
        dep = stack.pop()
        arcs.append((stack[-1], dep, action.split(":", 1)[1]))
    return stack, buffer, arcs

def oracle(sent):
    # statik orakul: etalon daraxtga qarab qaysi amalni bajarish kerakligini aytadi
    head = {t["id"]: t["head"] for t in sent}
    rel = {t["id"]: t["deprel"] for t in sent}
    n_deps = Counter(head.values())
    stack, buffer, arcs = [0], [t["id"] for t in sent], []
    steps = []
    while buffer or len(stack) > 1:
        done = Counter(h for h, _, _ in arcs)
        if len(stack) > 2 and head[stack[-2]] == stack[-1]:
            action = "LEFT-ARC:" + rel[stack[-2]]
        elif len(stack) >= 2 and head[stack[-1]] == stack[-2] and done[stack[-1]] == n_deps[stack[-1]]:
            action = "RIGHT-ARC:" + rel[stack[-1]]
        elif buffer:
            action = "SHIFT"
        else:
            raise ValueError("proektiv boʻlmagan daraxt: orakul uni qura olmaydi")
        steps.append((stack, buffer, arcs, action))
        stack, buffer, arcs = apply(action, stack, buffer, arcs)
    return steps, arcs

def show_trace(sent):
    form = {0: "ROOT", **{t["id"]: t["form"] for t in sent}}
    steps, arcs = oracle(sent)
    rows = []
    for k, (st, bf, ar, action) in enumerate(steps, 1):
        new = apply(action, st, bf, ar)[2][len(ar):]
        arc = f"{form[new[0][0]]} → {form[new[0][1]]}" if new else ""
        rows.append([k, " ".join(form[i] for i in st), " ".join(form[i] for i in bf), action, arc])
    print(pd.DataFrame(rows, columns=["qadam", "stek", "bufer", "amal", "yangi yoy"]).to_string(index=False))
    gold = sorted((t["head"], t["id"], t["deprel"]) for t in sent)
    print("\norakul qurgan daraxt etalon bilan mos keladi:", sorted(arcs) == gold)

show_trace(train[0])

Oʻzbek va rus tillarini solishtiramiz. Bosh soʻzi oxirida boʻlgan tilda bosh soʻz eng oxirida keladi, shuning uchun parser uzoq vaqt SHIFT qiladi, tobe soʻzlarni stekka yigʻadi, keyin ularni LEFT-ARC lar seriyasi bilan oladi. Stek chuqurroq boʻladi.

In [ ]:
def short(action):
    return {"SHIFT": "S", "LEFT-ARC": "L", "RIGHT-ARC": "R"}[action.split(":")[0]]

for lang, sents in [("oʻzbek", train), ("rus", other)]:
    depths = [max(len(st) for st, _, _, _ in oracle(s)[0]) - 1 for s in sents]
    print(f"{lang:<10} stekning maksimal chuqurligi (ROOT siz), gaplar boʻyicha oʻrtacha: {np.mean(depths):.2f}")
    for s in sents[:3]:
        steps, _ = oracle(s)
        print(f"   {' '.join(t['form'] for t in s):<46} {' '.join(short(a) for *_, a in steps)}")
    print()

Orakulga etalon javob kerak. Haqiqiy parser daraxtni bilmasdan, joriy konfiguratsiyaga qarab amalni taxmin qilishi kerak. Orakul izlarini «konfiguratsiya belgilari → amal» oʻqitish misollariga aylantiramiz va logistik regressiyani oʻqitamiz. Bu sxemani (klassifikatorli ochkoʻz parser) J. Nivre taklif qilgan; zamonaviy parserlarda logistik regressiya oʻrnida neyron tarmoq turadi, lekin sikl oʻsha.

Oʻzbek tili uchun soʻzning oxirgi harflari muhim belgi: «-ni» (tushum kelishigi) `obj` ni, «-da», «-ga», «-dan» `obl` ni koʻrsatadi.

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

def config_features(stack, buffer, arcs, sent):
    # konfiguratsiya belgilari: stekning ikki yuqori soʻzi va buferning birinchi soʻzining turkumi va oxirgi harflari, allaqachon ulangan chap tobe soʻzlar
    tok = {t["id"]: t for t in sent}
    def upos(i):
        return "<none>" if i is None else ("ROOT" if i == 0 else tok[i]["upos"])
    def suf(i):
        return "<none>" if i is None else ("ROOT" if i == 0 else tok[i]["form"].lower()[-2:])
    s0 = stack[-1] if stack else None
    s1 = stack[-2] if len(stack) >= 2 else None
    b0 = buffer[0] if buffer else None
    left = sorted(r for h, d, r in arcs if h == s0 and d < s0)
    return {
        "s0.upos": upos(s0), "s1.upos": upos(s1), "b0.upos": upos(b0),
        "s1+s0": upos(s1) + "+" + upos(s0),
        "s0.suf": suf(s0), "s1.suf": suf(s1),
        "s0.left": "|".join(left) or "-",
    }

X, y = [], []
for s in train:
    for stack, buffer, arcs, action in oracle(s)[0]:
        X.append(config_features(stack, buffer, arcs, s))
        y.append(action)
vec = DictVectorizer()
clf = LogisticRegression(max_iter=3000, C=10).fit(vec.fit_transform(X), y)
print(f"{len(X)} ta oʻqitish misoli (konfiguratsiya → amal)")
print(pd.Series(y).value_counts().to_string())

Parser ochkoʻz ishlaydi: har qadamda eng ehtimolli **ruxsat etilgan** amalni oladi va orqaga qaytmaydi (8-amaliy mashgʻulotdagi ochkoʻz dekoder kabi). Sifat toʻgʻri bosh soʻzli soʻzlar ulushi (UAS, unlabeled attachment score) va toʻgʻri bosh soʻz hamda bogʻlanish turli soʻzlar ulushi (LAS, labeled) bilan oʻlchanadi. Soʻz turkumlarini maʼlum deb hisoblaymiz (ularni 8-amaliy mashgʻulotdagi tegger berardi). Belgilar: ✓ — hammasi toʻgʻri, ~ — bosh soʻz toʻgʻri, lekin tur notoʻgʻri, ✗ — bosh soʻz notoʻgʻri.

In [ ]:
def legal(action, stack, buffer):
    # amalga ruxsat bormi? LEFT-ARC ni ROOT ga qoʻllab boʻlmaydi; ROOT ga RIGHT-ARC — faqat bufer boʻsh boʻlganda
    if action == "SHIFT":
        return bool(buffer)
    if action.startswith("LEFT-ARC"):
        return len(stack) > 2
    return len(stack) >= 2 and (stack[-2] != 0 or not buffer)

def parse(sent):
    stack, buffer, arcs = [0], [t["id"] for t in sent], []
    while buffer or len(stack) > 1:
        probs = clf.predict_proba(vec.transform([config_features(stack, buffer, arcs, sent)]))[0]
        for k in np.argsort(-probs):
            if legal(clf.classes_[k], stack, buffer):
                break
        stack, buffer, arcs = apply(clf.classes_[k], stack, buffer, arcs)
    return {d: (h, r) for h, d, r in arcs}

uas = las = n = 0
for s in test:
    pred = parse(s)
    form = {0: "ROOT", **{t["id"]: t["form"] for t in s}}
    print(" ".join(t["form"] for t in s))
    for t in s:
        h, r = pred[t["id"]]
        mark = "✓" if (h, r) == (t["head"], t["deprel"]) else ("~" if h == t["head"] else "✗")
        print(f"   {mark} {t['form']:<14} etalon: {form[t['head']]:<12} {t['deprel']:<10} parser: {form[h]:<12} {r}")
        uas += h == t["head"]
        las += (h, r) == (t["head"], t["deprel"])
        n += 1
print(f"\nUAS = {uas}/{n} = {uas / n:.2f}   LAS = {las}/{n} = {las / n:.2f}")

last = test[-1]
DependencyGraph(to_conllu_text(last, parse(last)), top_relation_label="root").tree().pretty_print()

Natijani tahlil qiling. Qoʻshimchalar parserga katta yordam beradi: «-ni» → `obj`, «-da», «-ga» → `obl`. Belgisiz toʻldiruvchi («xat yozdi», «suv ichyapti») va ega ikkalasi ham qoʻshimchasiz, ularni tartib ajratadi: feʼlga yaqinrogʻi — toʻldiruvchi. Feʼl stekka tushganda tobe soʻzlar oʻngdan chapga ulanadi, shuning uchun `s0.left` belgisi (feʼlda allaqachon `obj` bormi) bu yerda foydali.

Yagona xato — «Doʻstimning»: qaratqich kelishigidagi aniqlovchi (`nmod:poss`) oʻqitish maʼlumotlarida faqat bir marta uchraydi («Akamning kitobi…»). Parser «Doʻstimning singlisi» ni bogʻlash oʻrniga SHIFT qildi va oxirida «Doʻstimning» ni feʼlga ulab yubordi. Kam uchraydigan konstruksiyalarni kichik treebankdan oʻrganib boʻlmaydi.

## 3. Tarkibiy qismlar daraxtlari va CYK algoritmi

Tarkibiy qismlar (constituency) daraxti soʻzlarni ichma-ich iboralarga guruhlaydi: ot guruhi NP, feʼl guruhi VP. Qoidalarni kontekstdan erkin grammatika (CFG) beradi. CYK algoritmi **Xomskiy normal shaklini** (CNF) talab qiladi: har bir qoida $A \to B\,C$ yoki $A \to \text{soʻz}$ koʻrinishida.

CYK jadvalni pastdan yuqoriga toʻldiradi: $[i, j]$ katakda $i..j-1$ soʻzlarni hosil qiluvchi barcha noterminallar saqlanadi. Uzunligi 1 boʻlgan boʻlaklar uchun lugʻatga qaraymiz, uzunroqlari uchun $k$ boʻlish nuqtasini saralaymiz: agar $B \in [i,k]$, $C \in [k,j]$ va $A \to B\,C$ qoidasi boʻlsa, $A$ ni $[i,j]$ ga orqaga havola bilan qoʻshamiz. Murakkablik $O(n^3 \cdot |G|)$. Bu Viterbidagi dinamik dasturlashning oʻzi, faqat pozitsiyalar emas, boʻlaklar boʻyicha.

Oʻzbek grammatikasida ham bosh soʻz oxirida: VP → NP V (feʼl oxirida), izofa NP → NP Nposs (egalik qoʻshimchali bosh soʻz oxirida: «kitob javoni»). Misol: «akam eski kitob javonini sotdi». Nima eski: javonmi yoki kitoblarmi?

Qadamlar chop etilgandan keyin jadval rasmda chiziladi: yuqori oʻng katakda S, ikki usul bilan (×2).

In [ ]:
import nltk

GRAMMAR_TEXT = """
S     -> NP VP
VP    -> NP V
NP    -> Adj NP | NP Nposs | 'akam' | 'kitob' | 'maktab'
Adj   -> 'eski' | 'katta' | 'yangi'
Nposs -> 'javonini' | 'kutubxonasi'
V     -> 'sotdi'
"""

def read_grammar(text):
    # A -> B C | 'soʻz' koʻrinishidagi qoidalarni tahlil qilamiz
    binary, lexical = defaultdict(list), defaultdict(list)
    for line in text.strip().splitlines():
        lhs, rhs = [x.strip() for x in line.split("->")]
        for alt in rhs.split("|"):
            parts = alt.split()
            if len(parts) == 1 and parts[0][0] in "'\"":
                lexical[parts[0].strip("'\"")].append(lhs)
            else:
                binary[tuple(parts)].append(lhs)
    return binary, lexical

binary, lexical = read_grammar(GRAMMAR_TEXT)
print(f"grammatika Xomskiy normal shaklida: {nltk.CFG.fromstring(GRAMMAR_TEXT).is_chomsky_normal_form()}; "
      f"{sum(map(len, binary.values()))} ta binar qoida, {len(lexical)} ta soʻz lugʻatda\n")

def cyk(words, verbose=False):
    n = len(words)
    chart = defaultdict(lambda: defaultdict(list))     # chart[i, j][A] = [(k, B, C), ...]
    for i, w in enumerate(words):
        for A in lexical.get(w, []):
            chart[i, i + 1][A].append(w)
        if verbose:
            print(f"katak [{i},{i + 1}] {w}: {', '.join(chart[i, i + 1])}")
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length
            for k in range(i + 1, j):
                for B in list(chart[i, k]):
                    for C in list(chart[k, j]):
                        for A in binary.get((B, C), []):
                            chart[i, j][A].append((k, B, C))
            if verbose and chart[i, j]:
                found = "; ".join(f"{A} ← " + " | ".join(f"{B}[{i},{k}] + {C}[{k},{j}]" for k, B, C in bps)
                                  for A, bps in chart[i, j].items())
                print(f"katak [{i},{j}] {' '.join(words[i:j])}: {found}")
    return chart

def count(chart, i, j, A, memo=None):
    memo = {} if memo is None else memo
    if (i, j, A) not in memo:
        total = 0
        for bp in chart[i, j][A]:
            if isinstance(bp, str):
                total += 1
            else:
                k, B, C = bp
                total += count(chart, i, k, B, memo) * count(chart, k, j, C, memo)
        memo[i, j, A] = total
    return memo[i, j, A]

words = "akam eski kitob javonini sotdi".split()
chart = cyk(words, verbose=True)
print(f"\nS [0,{len(words)}]: {count(chart, 0, len(words), 'S')} ta tahlil")


n = len(words)
fig, ax = plt.subplots(figsize=(1.6 * n + 1, 0.9 * n + 1))
for i in range(n):
    for j in range(i + 1, n + 1):
        cell = chart[i, j]
        txt = "\n".join(f"{A}" + (f" ×{count(chart, i, j, A)}" if count(chart, i, j, A) > 1 else "") for A in cell)
        ax.add_patch(plt.Rectangle((j - 1.5, i - 0.5), 1, 1, fill=bool(cell), fc="#DCE4F7", ec="gray"))
        ax.text(j - 1, i, txt, ha="center", va="center", fontsize=9)
ax.set_xlim(-0.5, n - 0.5)
ax.set_ylim(n - 0.5, -0.5)
ax.set_xticks(range(n), words)
ax.set_yticks(range(n), [f"{i}: {w}" for i, w in enumerate(words)])
ax.set_xlabel("oxirgi soʻz")
ax.set_ylabel("boshi")
ax.set_title("CYK jadvali: [i, j] katak — i..j-1 soʻzlarni qamrab oluvchi tarkibiy qismlar (×k — k usul)", fontsize=9)
plt.tight_layout()
plt.show()

Orqaga havolalar boʻyicha barcha daraxtlarni tiklaymiz va ixtiyoriy CFG uchun kutubxona parseri `nltk.ChartParser` bilan solishtiramiz. Daraxtlarni `nltk.Tree.pretty_print()` chop etadi.

Birinchi daraxtda «eski kitob» birga turadi — eski kitoblar javoni. Ikkinchisida «eski» butun «kitob javoni» ga tegishli — javonning oʻzi eski.

In [ ]:
def build_trees(chart, i, j, A):
    out = []
    for bp in chart[i, j][A]:
        if isinstance(bp, str):
            out.append(nltk.Tree(A, [bp]))
        else:
            k, B, C = bp
            for left in build_trees(chart, i, k, B):
                for right in build_trees(chart, k, j, C):
                    out.append(nltk.Tree(A, [left, right]))
    return out

ours = build_trees(chart, 0, len(words), "S")
grammar = nltk.CFG.fromstring(GRAMMAR_TEXT)
theirs = list(nltk.ChartParser(grammar).parse(words))
print(f"bizning CYK: {len(ours)} ta tahlil;  nltk.ChartParser: {len(theirs)} ta tahlil;  "
      f"daraxtlar toʻplamlari mos keladi: {set(map(str, ours)) == set(map(str, theirs))}")
for t in theirs:
    t.pretty_print()

## 4. Koʻp maʼnolilikning oʻsishi va ehtimolli grammatika

Har bir yangi sifat ot guruhining istalgan darajasiga birikishi mumkin, shuning uchun tahlillar soni sifatlar soni bilan birga oʻsadi (bizning grammatikada chiziqli: 1, 2, 3, 4). Rus tilida esa vaziyat ogʻirroq: «я видел девушку с биноклем в парке…» da har bir yangi predlogli guruh oldingi istalgan NP ga yoki VP ga birikadi va tahlillar soni Katalan sonlari boʻyicha oʻsadi: 2, 5, 14, 42. Oʻzbek tilida morfologiya koʻpincha birikish joyini koʻrsatadi: «parkdagi qiz» (-dagi otga birikadi) va «parkda koʻrdim» (-da odatda feʼlga birikadi), «durbinli qiz» va «durbin bilan koʻrdim». Shuning uchun bunday koʻp maʼnolilik kamroq uchraydi.

**PCFG** (ehtimolli KE-grammatika) har bir qoidaga ehtimol beradi, daraxt ehtimoli — uning qoidalari ehtimollari koʻpaytmasi. `nltk.ViterbiParser` eng ehtimolli daraxtni xuddi shu CYK bilan topadi, faqat katakda noterminallar toʻplami oʻrniga ehtimol maksimumi saqlanadi.

In [ ]:
for s in [
    "akam kitob javonini sotdi",
    "akam eski kitob javonini sotdi",
    "akam eski katta kitob javonini sotdi",
    "akam eski maktab kutubxonasi javonini sotdi",
    "akam eski katta yangi kitob javonini sotdi",
]:
    w = s.split()
    ch = cyk(w)
    n_ours = count(ch, 0, len(w), "S")
    n_nltk = len(list(nltk.ChartParser(grammar).parse(w)))
    print(f"{len(w):2d} soʻz  {n_ours:4d} ta tahlil (nltk: {n_nltk:3d})   {s}")

pcfg = nltk.PCFG.fromstring("""
S     -> NP VP        [1.0]
VP    -> NP V         [1.0]
NP    -> Adj NP       [0.2]
NP    -> NP Nposs     [0.2]
NP    -> 'akam'       [0.3]
NP    -> 'kitob'      [0.2]
NP    -> 'maktab'     [0.1]
Adj   -> 'eski' [0.4] | 'katta' [0.3] | 'yangi' [0.3]
Nposs -> 'javonini' [0.5] | 'kutubxonasi' [0.5]
V     -> 'sotdi'      [1.0]
""")
prob = {(p.lhs(), p.rhs()): p.prob() for p in pcfg.productions()}
print()
for t in theirs:
    p = np.prod([prob[(r.lhs(), r.rhs())] for r in t.productions()])
    print(f"tahlil ehtimoli = {p:.6f}:  {t.pformat(margin=200)}")
best = next(nltk.ViterbiParser(pcfg).parse(words))
print(f"\nViterbiParser tanlaydi (p = {best.prob():.6f}):")
best.pretty_print()

Ikkala daraxt ehtimoli bir xil chiqdi: ikkalasida ham aynan bitta NP → Adj NP va bitta NP → NP Nposs qoidasi bor, faqat ular boshqa tartibda qoʻllangan. PCFG bu yerda tanlay olmaydi: ViterbiParser ulardan birini qaytaradi, lekin bu tanlov ehtimolga asoslanmagan. PCFG tuzilmani tanlashda soʻzlarga qaramaydi: «eski kitob javoni» va «yogʻoch kitob javoni» unga bir xil. Leksikalashtirilgan grammatikalar va neyron parserlar qaysi soʻzlar birikayotganini hisobga oladi.

## 5. Topshiriqlar

1. **Oʻz treebankingiz.** CoNLL-U formatida ikkita yangi gap yozing (biri `nmod:poss` bilan, masalan «Doʻstimning uyi daryo boʻyida turibdi» — «boʻyida» ni `obl` deb belgilang). `is_projective` ni tekshiring, `draw_tree` bilan chizing va `show_trace` ni ishga tushiring. Stek qanchalik chuqur boʻldi?
2. **Proektiv emaslik.** 1-boʻlimdagi gaplardan birida bitta bogʻliqlikning bosh soʻzini shunday oʻzgartiringki, yoylar kesishsin. `is_projective` `False` qaytarishiga va `oracle` xato bilan toʻxtashiga ishonch hosil qiling. Qaysi yoylar kesishadi?
3. **Parser belgilari.** `config_features` dagi `s0.suf` va `s1.suf` belgilarini olib tashlang, klassifikatorni qayta oʻqiting va UAS hamda LAS ni solishtiring. Qaysi bogʻlanish turlari adashtiriladi? Nega oʻzbek tili uchun qoʻshimchalar shunchalik muhim?
4. **Oʻz grammatikangiz.** `GRAMMAR_TEXT` ga yangi soʻzlar qoʻshing (masalan, `Adj -> 'yogʻoch'`, `Nposs -> 'eshigini'`) va «akam eski yogʻoch uy eshigini sotdi» kabi gapni tahlil qiling (kerak boʻlsa `NP -> 'uy'` ni ham qoʻshing). Nechta tahlil chiqdi? Grammatika CNF da qoladimi?
5. **PCFG.** «eski kitob» birikmasini afzal koʻradigan grammatika tuzing: masalan, alohida `NP -> AdjN Nposs` va `AdjN -> Adj NP` qoidalarini ehtimollari bilan qoʻshing. ViterbiParser endi qaysi daraxtni tanlaydi? Ikkala daraxt ehtimolini chop eting.

*Maslahat: Universal Dependencies dagi oʻzbek treebankini (UD_Uzbek-UT) yuklab olib, «Files» paneli orqali yuklashingiz va `read_conllu(open(...).read())` funksiyasi bilan oʻqishingiz mumkin.*

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing